# GoldML - Notebook Google Colab

Notebook này chia toàn bộ quy trình thành các cell độc lập để dễ chạy, giải thích và chụp kết quả đưa vào báo cáo.

**Mục tiêu:** dùng dữ liệu giá vàng từ Kaggle, xử lý tối đa 100.000 dòng, tạo 22 đặc trưng, huấn luyện Linear Regression và LSTM, sau đó so sánh kết quả.


## Cell 1 - Cài thư viện

Chạy cell này một lần đầu tiên. TensorFlow dùng để xây dựng mô hình LSTM; scikit-learn dùng cho PCA, chuẩn hóa và Linear Regression.

In [ ]:
!pip install -q numpy pandas scikit-learn tensorflow joblib matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## Cell 2 - Lấy mã nguồn pipeline

Cell này tải `pipeline.py` từ repository để notebook sử dụng đúng logic với chương trình `train.py` và web app. Nếu đã upload thư mục dự án vào Colab, có thể bỏ qua cell này và sửa đường dẫn trong `sys.path`.

In [ ]:
!rm -rf /content/goldslide
!git clone -q https://github.com/Ryng404/goldslide.git /content/goldslide

import sys
sys.path.insert(0, '/content/goldslide')
from pipeline import load_gold_csv, clean_data, make_features, train_linear, train_lstm
print('Đã nạp pipeline.py')


## Cell 3 - Upload dữ liệu Kaggle

Upload một file CSV có các cột tương đương `date/time`, `open`, `high`, `low`, `close`; `volume` là tùy chọn. Sau khi upload, notebook tự đặt tên file thành `/content/gold.csv`.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()
uploaded_name = next(iter(uploaded))
shutil.copy(uploaded_name, '/content/gold.csv')
print('File dữ liệu:', uploaded_name)


## Cell 4 - Đọc và kiểm tra dữ liệu gốc

Hàm đọc tự nhận dạng dấu phân cách, tên cột phổ biến và chuyển thời gian/số về đúng kiểu. Cell này giúp kiểm tra nhanh dữ liệu trước khi làm sạch.

In [ ]:
raw = load_gold_csv('/content/gold.csv')
print(f'Số dòng gốc: {len(raw):,}')
print(f'Thời gian: {raw.date.min()} đến {raw.date.max()}')
display(raw.head())
display(raw.describe(include='all').transpose())


## Cell 5 - Làm sạch dữ liệu

Các bước gồm xóa dòng thiếu, sắp xếp và bỏ trùng thời gian, kiểm tra OHLC, xử lý volume âm, loại outlier bằng MAD robust z-score trên log-return và giữ tối đa 100.000 dòng mới nhất.

In [ ]:
clean, cleaning_report = clean_data(raw, max_rows=100_000)
print(f'Số dòng sau làm sạch: {len(clean):,}')
display(cleaning_report)
display(clean.tail())


## Cell 6 - Quan sát giá sau làm sạch

Biểu đồ giúp phát hiện nhanh dữ liệu có bị đứt đoạn hoặc còn điểm bất thường lớn hay không.

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(clean['date'], clean['close'], color='#b45309', linewidth=0.8)
plt.title('Giá đóng cửa sau khi làm sạch')
plt.xlabel('Thời gian'); plt.ylabel('Giá vàng'); plt.grid(alpha=0.3)
plt.show()


## Cell 7 - Tạo 22 đặc trưng và biến mục tiêu

Đặc trưng gồm OHLCV, return nhiều kỳ, SMA ratio, biên độ nến, volatility, RSI, MACD và mã hóa chu kỳ thời gian bằng sin/cos. `target_return` là phần trăm thay đổi của giá đóng cửa ở phiên kế tiếp.

In [ ]:
data = make_features(clean)
feature_names = [
    'open', 'high', 'low', 'close', 'volume', 'return_1', 'return_5', 'return_20',
    'range_pct', 'body_pct', 'sma_ratio_5', 'sma_ratio_20', 'sma_ratio_50',
    'volatility_10', 'volatility_20', 'rsi_14', 'macd', 'macd_signal',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]
print(f'Số dòng dùng cho mô hình: {len(data):,}')
print(f'Số đặc trưng: {len(feature_names)}')
print('Biến mục tiêu: target_return')
display(data[feature_names + ['target_return']].head())


## Cell 8 - Chia train/test theo thời gian

Dùng 85% dữ liệu cũ để huấn luyện và 15% dữ liệu mới hơn để kiểm tra. Không shuffle vì không được để dữ liệu tương lai lọt vào quá trình học.

In [ ]:
TRAIN_RATIO = 0.85
split = int(len(data) * TRAIN_RATIO)
print(f'Train: {split:,} dòng ({TRAIN_RATIO:.0%})')
print(f'Test : {len(data) - split:,} dòng ({1 - TRAIN_RATIO:.0%})')
print(f'Mốc chia: {data.loc[split, "date"]}')


## Cell 9 - Huấn luyện Linear Regression với PCA

Linear Regression là mô hình đối chiếu dễ giải thích. Trong pipeline, 22 đặc trưng được chuẩn hóa rồi PCA giảm còn 12 thành phần trước khi huấn luyện.

In [ ]:
_, linear_metrics, linear_output = train_linear(data, split, pca_components=12)
print('Kết quả Linear Regression:')
for name, value in linear_metrics.items():
    print(f'  {name}: {value:.6f}')


## Cell 10 - Huấn luyện LSTM

Mỗi mẫu LSTM gồm 60 phiên liên tiếp × 22 đặc trưng để dự đoán return của phiên kế tiếp. LSTM là mô hình chính; EarlyStopping và ReduceLROnPlateau giúp hạn chế overfitting.

In [ ]:
lstm, lstm_metrics, lstm_output = train_lstm(
    data, split, window=60, epochs=40, batch_size=256
)
print('Kết quả LSTM:')
for name, value in lstm_metrics.items():
    print(f'  {name}: {value:.6f}')


## Cell 11 - Kiểm tra quá trình học của LSTM

Loss huấn luyện và validation loss nên giảm trong những epoch đầu. Nếu validation loss tăng trong khi training loss tiếp tục giảm, mô hình có dấu hiệu overfitting.

In [ ]:
history = lstm_output['history']
plt.figure(figsize=(10, 4))
plt.plot(history['loss'], label='Training loss')
plt.plot(history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Lịch sử huấn luyện LSTM')
plt.grid(alpha=0.3); plt.legend(); plt.show()


## Cell 12 - So sánh hai mô hình

So sánh trên cùng tập test. MAE và RMSE càng thấp càng tốt; R² cao hơn thường tốt hơn nhưng không nên dùng riêng R² để kết luận.

In [ ]:
results = pd.DataFrame({'Linear Regression': linear_metrics, 'LSTM': lstm_metrics}).T
display(results.style.format('{:.6f}'))
results[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4), color=['#2563eb', '#b45309'])
plt.title('So sánh sai số trên tập test')
plt.ylabel('Sai số'); plt.xticks(rotation=0); plt.grid(axis='y', alpha=0.3)
plt.show()


## Cell 13 - Biểu đồ dự đoán LSTM

Biểu đồ chỉ hiển thị 300 điểm kiểm tra cuối để dễ quan sát. Giá dự đoán được quy đổi từ return dự đoán về giá đóng cửa.

In [ ]:
n_show = min(300, len(lstm_output['dates']))
plt.figure(figsize=(14, 5))
plt.plot(lstm_output['dates'][-n_show:], lstm_output['actual'][-n_show:], label='Giá thật', color='black')
plt.plot(lstm_output['dates'][-n_show:], lstm_output['predicted'][-n_show:], label='LSTM', color='#b45309')
plt.xticks(rotation=30); plt.grid(alpha=0.3); plt.legend()
plt.title(f'{n_show} phiên cuối của tập test')
plt.show()


## Cell 14 - Lưu kết quả để dùng cho web app

Cell này tạo các file kết quả. Khi chạy trên máy tính, `train.py` cũng tạo các file tương tự trong thư mục `models/`.

In [ ]:
from pathlib import Path
import json
import joblib

output_dir = Path('/content/models')
output_dir.mkdir(exist_ok=True)
pd.DataFrame({
    'date': lstm_output['dates'],
    'actual': lstm_output['actual'],
    'predicted': lstm_output['predicted'],
}).to_csv(output_dir / 'predictions.csv', index=False)
data.to_csv(output_dir / 'features.csv', index=False)
cleaning_report.to_csv(output_dir / 'cleaning_report.csv', index=False)
with open(output_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'Linear Regression': linear_metrics, 'LSTM': lstm_metrics}, f, ensure_ascii=False, indent=2)
print('Đã lưu kết quả vào /content/models')


## Kết luận

Quy trình hoàn chỉnh là: đọc dữ liệu → làm sạch → tạo 22 đặc trưng → chia theo thời gian → huấn luyện Linear Regression và LSTM → đánh giá → lưu kết quả. Khi thuyết trình, cần nhấn mạnh LSTM dùng 60 phiên gần nhất và là mô hình chính của dự án.